# 05 — Pipeline LLM

OpenAI, Anthropic, Gemini, Groq, Cerebras, custom on_message, LLMLoop, tool-call protocol.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises LLM provider construction, fallback routing, ChatContext, and (T3) live completions.


### LLM provider construction


In [ ]:
import { OpenAILLM, AnthropicLLM, GroqLLM, CerebrasLLM, GoogleLLM } from "getpatter";
await cell('llm_providers', { tier: 1, env }, () => {
  const providers = [
    ['OpenAI',    new OpenAILLM({ apiKey: 'sk-test', model: 'gpt-4o-mini' })],
    ['Anthropic', new AnthropicLLM({ apiKey: 'sk-ant-test', model: 'claude-haiku-4-5-20251001' })],
    ['Groq',      new GroqLLM({ apiKey: 'gr-test', model: 'llama3-8b-8192' })],
    ['Cerebras',  new CerebrasLLM({ apiKey: 'ce-test', model: 'llama-4-scout-17b-16e-instruct' })],
    ['Google',    new GoogleLLM({ apiKey: 'go-test', model: 'gemini-2.0-flash' })],
  ] as const;
  for (const [name, p] of providers) console.log(`${name}: model=${(p as any).model}`);
});


### FallbackLLMProvider


In [ ]:
import { FallbackLLMProvider, OpenAILLM, AnthropicLLM } from "getpatter";
await cell('fallback_provider', { tier: 1, env }, () => {
  const fb = new FallbackLLMProvider([
    new OpenAILLM({ apiKey: 'sk-test', model: 'gpt-4o-mini' }),
    new AnthropicLLM({ apiKey: 'sk-ant-test', model: 'claude-haiku-4-5-20251001' }),
  ]);
  console.log(`FallbackLLMProvider with ${fb.providers.length} providers`);
});


### ChatContext


In [ ]:
import { ChatContext } from "getpatter";
await cell('chat_context', { tier: 1, env }, () => {
  const ctx = new ChatContext('You are a helpful assistant.');
  ctx.addUser('What is the capital of France?');
  ctx.addAssistant('The capital of France is Paris.');
  ctx.addUser('And Germany?');
  const msgs = ctx.getMessages();
  console.log(`Total messages: ${msgs.length}`);
  msgs.forEach(m => console.log(`  ${m.role}: ${m.content.slice(0, 50)}`));
});


### Live: OpenAI chat completion  *(T3 — requires `OPENAI_API_KEY`)*


In [ ]:
await cell('openai_chat_live', { tier: 3, required: ['openaiKey'], env }, async () => {
  const resp = await fetch('https://api.openai.com/v1/chat/completions', {
    method: 'POST',
    headers: { Authorization: `Bearer ${env.openaiKey}`, 'Content-Type': 'application/json' },
    body: JSON.stringify({ model: 'gpt-4o-mini', messages: [{ role: 'user', content: 'Reply: pong' }], max_tokens: 10 }),
  });
  const data = await resp.json() as any;
  console.log(`reply: ${data.choices[0].message.content}`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a call using the Pipeline engine with OpenAI LLM + tool call. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
await cell('live_preflight', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, () => {
  console.log(`  carrier:  Twilio ${env.twilioNumber}  →  ${env.targetNumber}`);
  console.log('  LLM:      OpenAI  (gpt-4o-mini)');
  console.log(`  webhook:  ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
});


### Live LLM call with tool *(T4)*


In [ ]:
import { Patter, Twilio, OpenAIRealtime, tool } from "getpatter";
await cell('live_llm_call', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, async () => {
  const getTime = tool({
    name: 'get_time',
    description: 'Return the current UTC time.',
    parameters: {},
    handler: () => new Date().toUTCString(),
  });
  const p = new Patter({
    carrier: new Twilio({ accountSid: env.twilioSid, authToken: env.twilioToken }),
    phoneNumber: env.twilioNumber,
    webhookUrl: env.publicWebhookUrl,
  });
  const agent = p.agent({
    systemPrompt: 'Demo assistant. Use the get_time tool if asked.',
    engine: new OpenAIRealtime({ apiKey: env.openaiKey }),
    tools: [getTime],
  });
  try {
    await p.call(env.targetNumber, { agent, firstMessage: 'Hello! Ask me the time.', ringTimeout: env.maxCallSeconds });
    console.log('✓ LLM call with tool completed');
  } finally {
    await hangupLeftoverCalls(env);
  }
});
